In [ ]:
# The following models are not in the order of their classification accuracies, best models are at the end of this notebook.
# Standard Parameters:
# BATCH_SIZE = 16
# EPOCHS = 20 
# LEARNING_RATE = 0.0001
# Image Size = 224x224
# If there is any change in these parameters, it will be explicitly mentioned in that particular cell.

In [50]:
# Necessary Imports
import os
import torch
from tqdm import tqdm
from torchvision import datasets
from torchvision.transforms import v2
from PIL import Image
import albumentations as A
from ultralytics import YOLO
from sklearn.metrics import classification_report
torch.manual_seed(37)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [49]:
# Testing Function
def final_test(model,labels, preprocess, test_path):
    model.eval()
    correct = 0
    total = 0
    correct_labels = []
    pred_labels = []
    
    results_summary = {}

    print("Running validation on subset...")
    
    for class_folder in os.listdir(test_path):
        folder_path = os.path.join(test_path, class_folder)
        if not os.path.isdir(folder_path): continue
        
        results_summary[class_folder] = {"correct": 0, "total": 0}
        
        for img_name in tqdm(os.listdir(folder_path), desc=f"Testing {class_folder}"):

            img_path = os.path.join(folder_path, img_name)
            correct_labels.append(class_folder.strip())
            
            try:
                img = Image.open(img_path).convert('RGB')
            except:
                continue
                
            input_tensor = preprocess(img).unsqueeze(0).to(DEVICE)
            
            with torch.no_grad():
                output = model(input_tensor)
                _, pred_idx = torch.max(output, 1)
                
                predicted_name = labels[pred_idx.item()]
                pred_labels.append(predicted_name.strip())
            
            if predicted_name.strip() == class_folder.strip():
                correct += 1
                results_summary[class_folder]["correct"] += 1
            
            total += 1
            results_summary[class_folder]["total"] += 1
    print(results_summary)

    print("\n" + "="*40)
    print(f"{'Class Name':<30} | {'Accuracy':<10}")
    print("-"*45)
    for cls, stats in results_summary.items():
        acc = (stats['correct']/stats['total'])*100 if stats['total'] > 0 else 0
        print(f"{cls:<30} | {acc:>8.2f}%")
    
    final_score = (correct / total) * 100
    print("="*40)
    print(f"OVERALL ACCURACY ON SUBSET: {final_score:.2f}%")
    return correct_labels, pred_labels

# **All Experiments**

In [45]:
# YOLO v11 + Efficient_Net_B4 (Pipeline)
# Trained on Plant_Doc + New_Plant_Diseases + Plant_Wild Dataset
# Tested on the dataset provided consisting 10 classes (excluding Tomato_Mosaic Virus, Tomato_Spider_Mites and Tomato_Target_Spot)

In [46]:
model_loaded = torch.load("Efficient_B4_Merged.pth",weights_only=False)

In [48]:
# Testing Block

full_dataset = datasets.ImageFolder("Merged_Dataset/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset/cropped_testing" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:02<00:00, 28.02it/s]

{'Potato_Early_blight': {'correct': 57, 'total': 87}, 'Potato_healthy': {'correct': 53, 'total': 63}, 'Potato_Lateblight': {'correct': 48, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 54, 'total': 100}, 'Tomato_Early_blight': {'correct': 31, 'total': 62}, 'Tomato_healthy': {'correct': 35, 'total': 71}, 'Tomato_Late_blight': {'correct': 50, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 45, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 49, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 64, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    65.52%
Potato_healthy                 |    84.13%
Potato_Lateblight              |    46.15%
Tomato_Bacterial_spot          |    54.00%
Tomato_Early_blight            |    50.00%
Tomato_healthy                 |    49.30%
Tomato_Late_blight             |    55.56%
Tomato_Leaf_mold               |    61.64%
Tomato_Septoria_leaf_s

In [ ]:
# YOLO v11 + Efficient_Net_B4 (Pipeline)
# Trained on Plant_Doc + New_Plant_Diseases + Plant_Wild Dataset, images were resized from 224x224 to 380x380
# Tested on the dataset provided consisting 10 classes (excluding Tomato_Mosaic Virus, Tomato_Spider_Mites and Tomato_Target_Spot)

In [ ]:
model_loaded = torch.load("Efficient_B4_Merged_380.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Merged_Dataset/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset/cropped_testing_380" 

final_test(model_loaded, labels,preprocess, TEST_DIR)

Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:02<00:00, 39.56it/s]

{'Potato_Early_blight': {'correct': 52, 'total': 87}, 'Potato_healthy': {'correct': 46, 'total': 63}, 'Potato_Lateblight': {'correct': 45, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 59, 'total': 100}, 'Tomato_Early_blight': {'correct': 29, 'total': 62}, 'Tomato_healthy': {'correct': 29, 'total': 71}, 'Tomato_Late_blight': {'correct': 50, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 57, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 41, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 51, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    59.77%
Potato_healthy                 |    73.02%
Potato_Lateblight              |    43.27%
Tomato_Bacterial_spot          |    59.00%
Tomato_Early_blight            |    46.77%
Tomato_healthy                 |    40.85%
Tomato_Late_blight             |    55.56%
Tomato_Leaf_mold               |    78.08%
Tomato_Septoria_leaf_s

In [ ]:
# YOLO v11 + Efficient_Net_B4 (Pipeline)
# Trained on Plant_Doc + Plant_Wild Dataset for 40 epochs
# Tested on the dataset provided consisting 10 classes (excluding Tomato_Mosaic Virus, Tomato_Spider_Mites and Tomato_Target_Spot)

In [ ]:
model_loaded = torch.load("Efficient_B4_Merged_Plant_Doc_Wild_Only.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_testing" 

final_test(model_loaded, labels, preprocess, TEST_DIR)

Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:02<00:00, 38.94it/s]

{'Potato_Early_blight': {'correct': 49, 'total': 87}, 'Potato_healthy': {'correct': 48, 'total': 63}, 'Potato_Lateblight': {'correct': 51, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 62, 'total': 100}, 'Tomato_Early_blight': {'correct': 35, 'total': 62}, 'Tomato_healthy': {'correct': 23, 'total': 71}, 'Tomato_Late_blight': {'correct': 45, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 53, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 46, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 56, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    56.32%
Potato_healthy                 |    76.19%
Potato_Lateblight              |    49.04%
Tomato_Bacterial_spot          |    62.00%
Tomato_Early_blight            |    56.45%
Tomato_healthy                 |    32.39%
Tomato_Late_blight             |    50.00%
Tomato_Leaf_mold               |    72.60%
Tomato_Septoria_leaf_s

In [ ]:
# YOLO v11 + Efficient_Net_B7 (Pipeline)
# Trained on Plant_Doc + Plant_Wild Dataset for 20 epochs with a lr=0.0001 and for another 20 epochs with a lr = 0.00005
# Tested on the dataset provided consisting 10 classes (excluding Tomato_Mosaic Virus, Tomato_Spider_Mites and Tomato_Target_Spot)

In [ ]:
model_loaded = torch.load("Efficient_B7_Plant_Doc_Wild.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_testing" 

final_test(model_loaded, labels, preprocess, TEST_DIR)

Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:03<00:00, 24.40it/s]

{'Potato_Early_blight': {'correct': 42, 'total': 87}, 'Potato_healthy': {'correct': 48, 'total': 63}, 'Potato_Lateblight': {'correct': 51, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 50, 'total': 100}, 'Tomato_Early_blight': {'correct': 29, 'total': 62}, 'Tomato_healthy': {'correct': 28, 'total': 71}, 'Tomato_Late_blight': {'correct': 41, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 51, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 52, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 53, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    48.28%
Potato_healthy                 |    76.19%
Potato_Lateblight              |    49.04%
Tomato_Bacterial_spot          |    50.00%
Tomato_Early_blight            |    46.77%
Tomato_healthy                 |    39.44%
Tomato_Late_blight             |    45.56%
Tomato_Leaf_mold               |    69.86%
Tomato_Septoria_leaf_s

In [ ]:
# Some other models that were tested:

# 1. YOLO v11 + ViT_L_16 (Trained on Plant_Doc, 20 epochs => lr = 0.0001 + 20 epochs => lr = 0.00005)
#    => Testing Accuracy = 28 % [On 28 classes]

# 2. YOLO v11 + Efficient_Net_B4 (Trained on Plant_Doc for 5 epochs)
#    => Testing Accuracy = 37.1 % [On 13 Classes]

# 3. YOLO v11 + Efficient_Net_B6 (Trained on Plant_Wild+Plant_Doc for 20 epochs)
#    => Testing Accuracy =  54.99 % [On 11 Classes]

In [ ]:
# YOLO v11 + Efficient_Net_B4 (Pipeline)
# Trained on Plant_Doc + Plant_Wild Dataset containing only Potato subclasses
# Tested on the dataset provided containing only subclasses concerning Potato

In [ ]:
model_loaded = torch.load("Potato-b4.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Separated_Merged_Dataset(Plant_Wild+Plant_Doc)/Potato/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Separated_Merged_Dataset(Plant_Wild+Plant_Doc)/Potato/cropped_testing" 

final_test(model_loaded,labels, preprocess, TEST_DIR)

Running validation on subset...


Testing Potato_Lateblight: 100%|██████████| 104/104 [00:02<00:00, 36.90it/s]

{'Potato_Early_blight': {'correct': 56, 'total': 87}, 'Potato_healthy': {'correct': 59, 'total': 63}, 'Potato_Lateblight': {'correct': 61, 'total': 104}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    64.37%
Potato_healthy                 |    93.65%
Potato_Lateblight              |    58.65%
OVERALL ACCURACY ON SUBSET: 69.29%


In [ ]:
# YOLO v11 + ConvexNet-Base (Pipeline)
# Trained on Plant_Doc + Plant_Wild Dataset containing only Potato subclasses
# Tested on the dataset provided containing only subclasses concerning Potato

In [ ]:
model_loaded = torch.load("Potato-ConVexNet-Base.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Separated_Merged_Dataset(Plant_Wild+Plant_Doc)/Potato/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Separated_Merged_Dataset(Plant_Wild+Plant_Doc)/Potato/cropped_testing" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

c:\Anaconda\envs\env\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Running validation on subset...


Testing Potato_Lateblight: 100%|██████████| 104/104 [00:01<00:00, 62.27it/s]

{'Potato_Early_blight': {'correct': 59, 'total': 87}, 'Potato_healthy': {'correct': 56, 'total': 63}, 'Potato_Lateblight': {'correct': 72, 'total': 104}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    67.82%
Potato_healthy                 |    88.89%
Potato_Lateblight              |    69.23%
OVERALL ACCURACY ON SUBSET: 73.62%


In [ ]:
# YOLO v11 + Efficient_Net_B7 (Pipeline)
# Trained on Plant_Doc + Plant_Wild Dataset containing only Tomato subclasses
# Tested on the dataset provided containing only subclasses concerning Tomato(excluding Spider mites, Target Spot and Mosaic Virus)

In [ ]:
model_loaded = torch.load("Tomato-b7.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Separated_Merged_Dataset(Plant_Wild+Plant_Doc)/Tomato/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Separated_Merged_Dataset(Plant_Wild+Plant_Doc)/Tomato/cropped_testing" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

c:\Anaconda\envs\env\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:03<00:00, 23.80it/s]

{'Tomato_Bacterial_spot': {'correct': 49, 'total': 100}, 'Tomato_Early_blight': {'correct': 32, 'total': 62}, 'Tomato_healthy': {'correct': 38, 'total': 71}, 'Tomato_Late_blight': {'correct': 55, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 55, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 64, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 59, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Tomato_Bacterial_spot          |    49.00%
Tomato_Early_blight            |    51.61%
Tomato_healthy                 |    53.52%
Tomato_Late_blight             |    61.11%
Tomato_Leaf_mold               |    75.34%
Tomato_Septoria_leaf_spot      |    59.26%
Tomato_Tomato_Yellow_Leaf_Curl_Virus |    70.24%
OVERALL ACCURACY ON SUBSET: 59.86%


In [ ]:
# YOLO v11 + ConvexNet-Base (Pipeline)
# Trained on Plant_Doc + Plant_Wild Dataset containing only Tomato subclasses
# Tested on the dataset provided containing only subclasses concerning Tomato(excluding Spider mites, Target Spot and Mosaic Virus)

In [ ]:
model_loaded = torch.load("Tomato-ConVexNet-Base.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Separated_Merged_Dataset(Plant_Wild+Plant_Doc)/Tomato/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Separated_Merged_Dataset(Plant_Wild+Plant_Doc)/Tomato/cropped_testing" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

c:\Anaconda\envs\env\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:01<00:00, 65.53it/s]

{'Tomato_Bacterial_spot': {'correct': 51, 'total': 100}, 'Tomato_Early_blight': {'correct': 40, 'total': 62}, 'Tomato_healthy': {'correct': 38, 'total': 71}, 'Tomato_Late_blight': {'correct': 61, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 60, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 67, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 74, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Tomato_Bacterial_spot          |    51.00%
Tomato_Early_blight            |    64.52%
Tomato_healthy                 |    53.52%
Tomato_Late_blight             |    67.78%
Tomato_Leaf_mold               |    82.19%
Tomato_Septoria_leaf_spot      |    62.04%
Tomato_Tomato_Yellow_Leaf_Curl_Virus |    88.10%
OVERALL ACCURACY ON SUBSET: 66.50%


In [ ]:
# YOLO v11 + ConvexNet-Large (Pipeline)
# Trained on Plant_Doc + Plant_Wild Dataset for 10 epochs
# Tested on the dataset provided consisting 10 classes (excluding Tomato_Mosaic Virus, Tomato_Spider_Mites and Tomato_Target_Spot)

In [ ]:
model_loaded = torch.load("ConVexNet-Large-Merged-Plant_Doc_Wild.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_testing" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

c:\Anaconda\envs\env\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:01<00:00, 63.52it/s]

{'Potato_Early_blight': {'correct': 37, 'total': 87}, 'Potato_healthy': {'correct': 37, 'total': 63}, 'Potato_Lateblight': {'correct': 59, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 53, 'total': 100}, 'Tomato_Early_blight': {'correct': 45, 'total': 62}, 'Tomato_healthy': {'correct': 35, 'total': 71}, 'Tomato_Late_blight': {'correct': 45, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 56, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 51, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 60, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    42.53%
Potato_healthy                 |    58.73%
Potato_Lateblight              |    56.73%
Tomato_Bacterial_spot          |    53.00%
Tomato_Early_blight            |    72.58%
Tomato_healthy                 |    49.30%
Tomato_Late_blight             |    50.00%
Tomato_Leaf_mold               |    76.71%
Tomato_Septoria_leaf_s

In [ ]:
# YOLO v11 + ConvexNet-Base (Pipeline)
# Trained on Plant_Doc + Plant_Wild Dataset for 20 epochs, and included a new transformation during training transforms.equalize()
# Tried resizing the images from 224x224 to 299x299
# Tested on the dataset provided consisting 10 classes (excluding Tomato_Mosaic Virus, Tomato_Spider_Mites and Tomato_Target_Spot)

In [ ]:
model_loaded = torch.load("ConVexNet-Base-Merged-Plant_Doc_Wild_299.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_training_299")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_testing_299" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

c:\Anaconda\envs\env\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:01<00:00, 60.87it/s]

{'Potato_Early_blight': {'correct': 34, 'total': 87}, 'Potato_healthy': {'correct': 43, 'total': 63}, 'Potato_Lateblight': {'correct': 55, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 77, 'total': 100}, 'Tomato_Early_blight': {'correct': 30, 'total': 62}, 'Tomato_healthy': {'correct': 43, 'total': 71}, 'Tomato_Late_blight': {'correct': 56, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 57, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 44, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 65, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    39.08%
Potato_healthy                 |    68.25%
Potato_Lateblight              |    52.88%
Tomato_Bacterial_spot          |    77.00%
Tomato_Early_blight            |    48.39%
Tomato_healthy                 |    60.56%
Tomato_Late_blight             |    62.22%
Tomato_Leaf_mold               |    78.08%
Tomato_Septoria_leaf_s

In [ ]:
# YOLO v11 + ConvexNet-Small (Pipeline)
# Used the small model to make sure ConvexNet-Base was not overfitting the training set
# Trained on Plant_Doc + Plant_Wild Dataset for 20 epochs, and included a new transformation during training transforms.equalize()
# Tested on the dataset provided consisting 10 classes (excluding Tomato_Mosaic Virus, Tomato_Spider_Mites and Tomato_Target_Spot)

In [ ]:
model_loaded = torch.load("ConVexNet-Small-Merged-Plant_Doc_Wild.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_testing" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

c:\Anaconda\envs\env\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:05<00:00, 15.73it/s]

{'Potato_Early_blight': {'correct': 48, 'total': 87}, 'Potato_healthy': {'correct': 41, 'total': 63}, 'Potato_Lateblight': {'correct': 47, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 54, 'total': 100}, 'Tomato_Early_blight': {'correct': 36, 'total': 62}, 'Tomato_healthy': {'correct': 36, 'total': 71}, 'Tomato_Late_blight': {'correct': 61, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 48, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 50, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 45, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    55.17%
Potato_healthy                 |    65.08%
Potato_Lateblight              |    45.19%
Tomato_Bacterial_spot          |    54.00%
Tomato_Early_blight            |    58.06%
Tomato_healthy                 |    50.70%
Tomato_Late_blight             |    67.78%
Tomato_Leaf_mold               |    65.75%
Tomato_Septoria_leaf_s

In [ ]:
# -----------------------------------------------------------------------------RICE DISEASES---------------------------------------------------------------------------------

In [ ]:
# Efficient Net B4
# Trained on Rice Disease Dataset shared with us with basic transformations applied for 20 epochs
# Tested on the dataset provided to us to verify accuracy

In [ ]:
model_loaded = torch.load("RiceDisease_EfficientNetB4_Baseline_20_Epochs.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Rice_Disease_Dataset/Rice Disease Dataset_split/train")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.Resize((224,224)),
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Rice_Disease_Dataset/Rice Disease Dataset_split/test" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

Running validation on subset...


Testing Tungro: 100%|██████████| 30/30 [00:01<00:00, 29.15it/s]

{'Bacterial Leaf Blight': {'correct': 30, 'total': 30}, 'Bacterial Streak': {'correct': 15, 'total': 15}, 'Bakanae': {'correct': 15, 'total': 15}, 'Brown Spot': {'correct': 22, 'total': 30}, 'False Smut': {'correct': 15, 'total': 15}, 'Grassy Stunt Virus': {'correct': 12, 'total': 15}, 'Healthy Leaf': {'correct': 30, 'total': 30}, 'Hispa': {'correct': 28, 'total': 30}, 'Insect Affected': {'correct': 29, 'total': 30}, 'Leaf Blast': {'correct': 30, 'total': 30}, 'Leaf Scald': {'correct': 29, 'total': 30}, 'Leaf Smut': {'correct': 8, 'total': 30}, 'Narrow Brown Spot': {'correct': 30, 'total': 30}, 'Neck Blast': {'correct': 30, 'total': 30}, 'Ragged Stunt Virus': {'correct': 13, 'total': 15}, 'Sheath Blight': {'correct': 30, 'total': 30}, 'Sheath Rot': {'correct': 6, 'total': 15}, 'Stem Rot': {'correct': 15, 'total': 15}, 'Tungro': {'correct': 30, 'total': 30}}

Class Name                     | Accuracy  
---------------------------------------------
Bacterial Leaf Blight          |   100.

In [ ]:
# ConvexNet-Base
# Trained on Rice Disease Dataset shared with us with basic transformations along with equalization applied for 20 epochs
# Tested on the dataset provided to us to verify accuracy

In [ ]:
model_loaded = torch.load("ConvexNet-Base-Rice-Disease-20-Epoch-Equalized.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Rice_Disease_Dataset/Rice Disease Dataset_split/train")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.Resize((224,224)),
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Rice_Disease_Dataset/Rice Disease Dataset_split/test" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

Running validation on subset...


Testing Tungro: 100%|██████████| 30/30 [00:01<00:00, 28.78it/s]

{'Bacterial Leaf Blight': {'correct': 28, 'total': 30}, 'Bacterial Streak': {'correct': 14, 'total': 15}, 'Bakanae': {'correct': 15, 'total': 15}, 'Brown Spot': {'correct': 22, 'total': 30}, 'False Smut': {'correct': 15, 'total': 15}, 'Grassy Stunt Virus': {'correct': 12, 'total': 15}, 'Healthy Leaf': {'correct': 30, 'total': 30}, 'Hispa': {'correct': 29, 'total': 30}, 'Insect Affected': {'correct': 28, 'total': 30}, 'Leaf Blast': {'correct': 30, 'total': 30}, 'Leaf Scald': {'correct': 30, 'total': 30}, 'Leaf Smut': {'correct': 24, 'total': 30}, 'Narrow Brown Spot': {'correct': 26, 'total': 30}, 'Neck Blast': {'correct': 30, 'total': 30}, 'Ragged Stunt Virus': {'correct': 12, 'total': 15}, 'Sheath Blight': {'correct': 30, 'total': 30}, 'Sheath Rot': {'correct': 5, 'total': 15}, 'Stem Rot': {'correct': 15, 'total': 15}, 'Tungro': {'correct': 30, 'total': 30}}

Class Name                     | Accuracy  
---------------------------------------------
Bacterial Leaf Blight          |    93

In [ ]:
# ConvexNet-Base
# Trained on Rice Disease Dataset shared with us with basic transformations along with Center-Cropping applied for 20 epochs
# Tested on the dataset provided to us to verify accuracy

In [ ]:
model_loaded = torch.load("ConvexNet-Base-Rice-Disease-20-Epoch-Center-Crop.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Rice_Disease_Dataset/Rice Disease Dataset_split/train")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.Resize((224,224)),
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Rice_Disease_Dataset/Rice Disease Dataset_split/test" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

Running validation on subset...


Testing Tungro: 100%|██████████| 30/30 [00:00<00:00, 42.88it/s]

{'Bacterial Leaf Blight': {'correct': 30, 'total': 30}, 'Bacterial Streak': {'correct': 8, 'total': 15}, 'Bakanae': {'correct': 15, 'total': 15}, 'Brown Spot': {'correct': 22, 'total': 30}, 'False Smut': {'correct': 15, 'total': 15}, 'Grassy Stunt Virus': {'correct': 14, 'total': 15}, 'Healthy Leaf': {'correct': 30, 'total': 30}, 'Hispa': {'correct': 30, 'total': 30}, 'Insect Affected': {'correct': 29, 'total': 30}, 'Leaf Blast': {'correct': 29, 'total': 30}, 'Leaf Scald': {'correct': 30, 'total': 30}, 'Leaf Smut': {'correct': 2, 'total': 30}, 'Narrow Brown Spot': {'correct': 29, 'total': 30}, 'Neck Blast': {'correct': 30, 'total': 30}, 'Ragged Stunt Virus': {'correct': 15, 'total': 15}, 'Sheath Blight': {'correct': 30, 'total': 30}, 'Sheath Rot': {'correct': 8, 'total': 15}, 'Stem Rot': {'correct': 15, 'total': 15}, 'Tungro': {'correct': 30, 'total': 30}}

Class Name                     | Accuracy  
---------------------------------------------
Bacterial Leaf Blight          |   100.0

In [ ]:
# ConvexNet-Large
# Trained on Rice Disease Dataset shared with us with basic transformations applied for 20 epochs
# Tested on the dataset provided to us to verify accuracy

In [ ]:
model_loaded = torch.load("ConvexNet-Large-Rice-Disease-15-Epochs.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Rice_Disease_Dataset/Rice Disease Dataset_split/train")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.Resize((224,224)),
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Rice_Disease_Dataset/Rice Disease Dataset_split/test" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

Running validation on subset...


Testing Tungro: 100%|██████████| 30/30 [00:00<00:00, 38.15it/s]

{'Bacterial Leaf Blight': {'correct': 28, 'total': 30}, 'Bacterial Streak': {'correct': 15, 'total': 15}, 'Bakanae': {'correct': 15, 'total': 15}, 'Brown Spot': {'correct': 23, 'total': 30}, 'False Smut': {'correct': 15, 'total': 15}, 'Grassy Stunt Virus': {'correct': 12, 'total': 15}, 'Healthy Leaf': {'correct': 30, 'total': 30}, 'Hispa': {'correct': 28, 'total': 30}, 'Insect Affected': {'correct': 29, 'total': 30}, 'Leaf Blast': {'correct': 30, 'total': 30}, 'Leaf Scald': {'correct': 29, 'total': 30}, 'Leaf Smut': {'correct': 2, 'total': 30}, 'Narrow Brown Spot': {'correct': 29, 'total': 30}, 'Neck Blast': {'correct': 30, 'total': 30}, 'Ragged Stunt Virus': {'correct': 11, 'total': 15}, 'Sheath Blight': {'correct': 30, 'total': 30}, 'Sheath Rot': {'correct': 8, 'total': 15}, 'Stem Rot': {'correct': 15, 'total': 15}, 'Tungro': {'correct': 30, 'total': 30}}

Class Name                     | Accuracy  
---------------------------------------------
Bacterial Leaf Blight          |    93.

In [ ]:
# ConvexNet-Base
# Trained on Rice Disease Dataset shared with us with basic transformations and weighted sampling applied for 10 epochs
# Tested on the dataset provided to us to verify accuracy

In [ ]:
model_loaded = torch.load("ConvexNet-Base-Rice-Disease-Weighted-10-Epochs.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Rice_Disease_Dataset/Rice Disease Dataset_split/train")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.Resize((224,224)),
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Rice_Disease_Dataset/Rice Disease Dataset_split/test" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

Running validation on subset...


Testing Tungro: 100%|██████████| 30/30 [00:01<00:00, 15.26it/s]

{'Bacterial Leaf Blight': {'correct': 28, 'total': 30}, 'Bacterial Streak': {'correct': 15, 'total': 15}, 'Bakanae': {'correct': 15, 'total': 15}, 'Brown Spot': {'correct': 22, 'total': 30}, 'False Smut': {'correct': 14, 'total': 15}, 'Grassy Stunt Virus': {'correct': 13, 'total': 15}, 'Healthy Leaf': {'correct': 30, 'total': 30}, 'Hispa': {'correct': 30, 'total': 30}, 'Insect Affected': {'correct': 29, 'total': 30}, 'Leaf Blast': {'correct': 29, 'total': 30}, 'Leaf Scald': {'correct': 29, 'total': 30}, 'Leaf Smut': {'correct': 28, 'total': 30}, 'Narrow Brown Spot': {'correct': 29, 'total': 30}, 'Neck Blast': {'correct': 30, 'total': 30}, 'Ragged Stunt Virus': {'correct': 11, 'total': 15}, 'Sheath Blight': {'correct': 30, 'total': 30}, 'Sheath Rot': {'correct': 10, 'total': 15}, 'Stem Rot': {'correct': 15, 'total': 15}, 'Tungro': {'correct': 30, 'total': 30}}

Class Name                     | Accuracy  
---------------------------------------------
Bacterial Leaf Blight          |    9

In [ ]:
# Inception V3
# Trained on Rice Disease Dataset shared with us with basic transformations, weighted sampling and after resizing the images to 299x299 for 30 epochs
# Tested on the dataset provided to us to verify accuracy

In [ ]:
model_loaded = torch.load("InceptionV3-Rice-Disease-Weighted-30-Epochs.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Rice_Disease_Dataset/Rice Disease Dataset_split/train")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.Resize((299,299)),
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Rice_Disease_Dataset/Rice Disease Dataset_split/test" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

Running validation on subset...


Testing Tungro: 100%|██████████| 30/30 [00:00<00:00, 37.22it/s]

{'Bacterial Leaf Blight': {'correct': 28, 'total': 30}, 'Bacterial Streak': {'correct': 15, 'total': 15}, 'Bakanae': {'correct': 15, 'total': 15}, 'Brown Spot': {'correct': 23, 'total': 30}, 'False Smut': {'correct': 15, 'total': 15}, 'Grassy Stunt Virus': {'correct': 12, 'total': 15}, 'Healthy Leaf': {'correct': 29, 'total': 30}, 'Hispa': {'correct': 28, 'total': 30}, 'Insect Affected': {'correct': 27, 'total': 30}, 'Leaf Blast': {'correct': 26, 'total': 30}, 'Leaf Scald': {'correct': 29, 'total': 30}, 'Leaf Smut': {'correct': 27, 'total': 30}, 'Narrow Brown Spot': {'correct': 29, 'total': 30}, 'Neck Blast': {'correct': 30, 'total': 30}, 'Ragged Stunt Virus': {'correct': 14, 'total': 15}, 'Sheath Blight': {'correct': 28, 'total': 30}, 'Sheath Rot': {'correct': 10, 'total': 15}, 'Stem Rot': {'correct': 13, 'total': 15}, 'Tungro': {'correct': 30, 'total': 30}}

Class Name                     | Accuracy  
---------------------------------------------
Bacterial Leaf Blight          |    9

In [ ]:
# VGG 16
# Trained on Rice Disease Dataset shared with us with basic transformations, weighted sampling for 30 epochs
# Tested on the dataset provided to us to verify accuracy

In [ ]:
model_loaded = torch.load("VGG16-Rice-Disease-Weighted-30-Epochs.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Rice_Disease_Dataset/Rice Disease Dataset_split/train")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.Resize((224,224)),
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Rice_Disease_Dataset/Rice Disease Dataset_split/test" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

Running validation on subset...


Testing Tungro: 100%|██████████| 30/30 [00:00<00:00, 56.26it/s]

{'Bacterial Leaf Blight': {'correct': 29, 'total': 30}, 'Bacterial Streak': {'correct': 15, 'total': 15}, 'Bakanae': {'correct': 15, 'total': 15}, 'Brown Spot': {'correct': 14, 'total': 30}, 'False Smut': {'correct': 15, 'total': 15}, 'Grassy Stunt Virus': {'correct': 12, 'total': 15}, 'Healthy Leaf': {'correct': 25, 'total': 30}, 'Hispa': {'correct': 30, 'total': 30}, 'Insect Affected': {'correct': 27, 'total': 30}, 'Leaf Blast': {'correct': 26, 'total': 30}, 'Leaf Scald': {'correct': 27, 'total': 30}, 'Leaf Smut': {'correct': 25, 'total': 30}, 'Narrow Brown Spot': {'correct': 28, 'total': 30}, 'Neck Blast': {'correct': 30, 'total': 30}, 'Ragged Stunt Virus': {'correct': 11, 'total': 15}, 'Sheath Blight': {'correct': 30, 'total': 30}, 'Sheath Rot': {'correct': 13, 'total': 15}, 'Stem Rot': {'correct': 15, 'total': 15}, 'Tungro': {'correct': 30, 'total': 30}}

Class Name                     | Accuracy  
---------------------------------------------
Bacterial Leaf Blight          |    9

In [ ]:
# VGG 16
# Trained on Rice Disease Dataset shared with us with basic transformations for 30 epochs
# Tested on the dataset provided to us to verify accuracy

In [ ]:
model_loaded = torch.load("VGG16-Rice-Disease-30-Epochs.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Rice_Disease_Dataset/Rice Disease Dataset_split/train")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.Resize((224,224)),
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Rice_Disease_Dataset/Rice Disease Dataset_split/test" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

Running validation on subset...


Testing Tungro: 100%|██████████| 30/30 [00:00<00:00, 50.85it/s]

{'Bacterial Leaf Blight': {'correct': 27, 'total': 30}, 'Bacterial Streak': {'correct': 15, 'total': 15}, 'Bakanae': {'correct': 15, 'total': 15}, 'Brown Spot': {'correct': 22, 'total': 30}, 'False Smut': {'correct': 12, 'total': 15}, 'Grassy Stunt Virus': {'correct': 11, 'total': 15}, 'Healthy Leaf': {'correct': 29, 'total': 30}, 'Hispa': {'correct': 26, 'total': 30}, 'Insect Affected': {'correct': 24, 'total': 30}, 'Leaf Blast': {'correct': 28, 'total': 30}, 'Leaf Scald': {'correct': 30, 'total': 30}, 'Leaf Smut': {'correct': 14, 'total': 30}, 'Narrow Brown Spot': {'correct': 25, 'total': 30}, 'Neck Blast': {'correct': 30, 'total': 30}, 'Ragged Stunt Virus': {'correct': 12, 'total': 15}, 'Sheath Blight': {'correct': 27, 'total': 30}, 'Sheath Rot': {'correct': 4, 'total': 15}, 'Stem Rot': {'correct': 9, 'total': 15}, 'Tungro': {'correct': 29, 'total': 30}}

Class Name                     | Accuracy  
---------------------------------------------
Bacterial Leaf Blight          |    90.

In [ ]:
# YOLOv11 + ConvexNet-Base
# Trained on Plant_Doc+Plant_Wild dataset with basic transformations and applying equalization for 20 epochs
# Tested on the test set provided by Plant_Doc to verify accuracy

In [ ]:
model_loaded = torch.load("ConvexNet-Base-Merged-Plant-Doc-Wild_27_Classes.pth",weights_only=False)

In [ ]:
# Testing Block

full_dataset = datasets.ImageFolder("Merged_Dataset[Plant_Doc+Plant_Wild][All Classes]/train")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.Resize((224,224)),
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset[Plant_Doc+Plant_Wild][All Classes]/test" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

Running validation on subset...


Testing Tomato Septoria leaf spot: 100%|██████████| 11/11 [00:00<00:00, 21.26it/s]

{'Apple leaf': {'correct': 6, 'total': 9}, 'Apple rust leaf': {'correct': 8, 'total': 10}, 'Apple Scab Leaf': {'correct': 9, 'total': 10}, 'Bell_pepper leaf': {'correct': 8, 'total': 8}, 'Bell_pepper leaf spot': {'correct': 8, 'total': 9}, 'Blueberry leaf': {'correct': 9, 'total': 11}, 'Cherry leaf': {'correct': 8, 'total': 10}, 'Corn Gray leaf spot': {'correct': 2, 'total': 4}, 'Corn leaf blight': {'correct': 4, 'total': 12}, 'Corn rust leaf': {'correct': 9, 'total': 10}, 'grape leaf': {'correct': 12, 'total': 12}, 'grape leaf black rot': {'correct': 7, 'total': 8}, 'Peach leaf': {'correct': 6, 'total': 9}, 'Potato leaf early blight': {'correct': 4, 'total': 8}, 'Potato leaf late blight': {'correct': 3, 'total': 8}, 'Raspberry leaf': {'correct': 5, 'total': 7}, 'Soyabean leaf': {'correct': 6, 'total': 8}, 'Squash Powdery mildew leaf': {'correct': 6, 'total': 6}, 'Strawberry leaf': {'correct': 8, 'total': 8}, 'Tomato Early blight leaf': {'correct': 7, 'total': 9}, 'Tomato leaf': {'corr

In [ ]:
# YOLOv11[For Cropping] + YOLO26-L[For Classification]
# Trained on Plant_Doc+Plant_Wild dataset with basic transformations and applying equalization for 100 epochs with patience=30
# Tested on the test set provided by Plant_Doc to verify accuracy

In [ ]:
# Testing Block
test_transform = [
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]

model_loaded = YOLO("runs/classify/train9/weights/best.pt")
metrics_testing = model_loaded.val(data="Merged_Dataset[Plant_Doc+Plant_Wild][All Classes]/train_split",split='test',device=0,augmentations=test_transform)
print(metrics_testing.top1)
print(metrics_testing.top5)

Ultralytics 8.4.21  Python-3.11.14 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3070 Ti, 8192MiB)
YOLO26l-cls summary (fused): 94 layers, 12,853,019 parameters, 0 gradients, 49.4 GFLOPs
train: C:\Users\Anuraag Shukla\Downloads\Vision Model Selection\Merged_Dataset[Plant_Doc+Plant_Wild][All Classes]\train_split\train... found 7110 images in 27 classes  
val: C:\Users\Anuraag Shukla\Downloads\Vision Model Selection\Merged_Dataset[Plant_Doc+Plant_Wild][All Classes]\train_split\val... found 1789 images in 27 classes  
test: C:\Users\Anuraag Shukla\Downloads\Vision Model Selection\Merged_Dataset[Plant_Doc+Plant_Wild][All Classes]\train_split\test... found 236 images in 27 classes  
test: Fast image access  (ping: 0.20.1 ms, read: 3.80.9 MB/s, size: 22.1 KB)
test: Scanning C:\Users\Anuraag Shukla\Downloads\Vision Model Selection\Merged_Dataset[Plant_Doc+Plant_Wild][All Classes]\train_split\test... 236 images, 0 corrupt: 100% ━━━━━━━━━━━━ 236/236  0.0s
               classes   top1_acc   top

In [ ]:
# YOLOv11[For Cropping] + YOLO26-X[For Classification]
# Trained on Plant_Doc+Plant_Wild dataset with basic transformations and applying equalization for 100 epochs with patience=30
# Tested on the test set provided by Plant_Doc to verify accuracy

In [ ]:
# Testing Block
test_transform = [
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]

model_loaded = YOLO("runs/classify/train10/weights/best.pt")
metrics_testing = model_loaded.val(data="Merged_Dataset[Plant_Doc+Plant_Wild][All Classes]/train_split",split='test',device=0,augmentations=test_transform)
print(metrics_testing.top1)
print(metrics_testing.top5)

Ultralytics 8.4.21  Python-3.11.14 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3070 Ti, 8192MiB)
YOLO26x-cls summary (fused): 94 layers, 28,367,003 parameters, 0 gradients, 110.4 GFLOPs
train: C:\Users\Anuraag Shukla\Downloads\Vision Model Selection\Merged_Dataset[Plant_Doc+Plant_Wild][All Classes]\train_split\train... found 7110 images in 27 classes  
val: C:\Users\Anuraag Shukla\Downloads\Vision Model Selection\Merged_Dataset[Plant_Doc+Plant_Wild][All Classes]\train_split\val... found 1789 images in 27 classes  
test: C:\Users\Anuraag Shukla\Downloads\Vision Model Selection\Merged_Dataset[Plant_Doc+Plant_Wild][All Classes]\train_split\test... found 236 images in 27 classes  
test: Fast image access  (ping: 0.00.0 ms, read: 223.672.8 MB/s, size: 22.1 KB)
test: Scanning C:\Users\Anuraag Shukla\Downloads\Vision Model Selection\Merged_Dataset[Plant_Doc+Plant_Wild][All Classes]\train_split\test... 236 images, 0 corrupt: 100% ━━━━━━━━━━━━ 236/236  0.0s
               classes   top1_acc  

In [ ]:
# YOLOv11[For Cropping] + YOLO26-L[For Classification]
# Trained on Rice Disease dataset with basic transformations for 100 epochs with patience=30
# Tested on the test set provided to us to verify accuracy

In [ ]:
# Testing Block
test_transform = [
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]

model_loaded = YOLO("runs/classify/train11/weights/best.pt")
metrics_testing = model_loaded.val(data="Rice_Disease_Dataset/Rice Disease Dataset_split/",split='test',device=0,augmentations=test_transform)
print(metrics_testing.top1)
print(metrics_testing.top5)

Ultralytics 8.4.21  Python-3.11.14 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3070 Ti, 8192MiB)
YOLO26l-cls summary (fused): 94 layers, 12,842,771 parameters, 0 gradients, 49.3 GFLOPs
train: C:\Users\Anuraag Shukla\Downloads\Vision Model Selection\Rice_Disease_Dataset\Rice Disease Dataset_split\train... found 36338 images in 19 classes  
val: C:\Users\Anuraag Shukla\Downloads\Vision Model Selection\Rice_Disease_Dataset\Rice Disease Dataset_split\val... found 9091 images in 19 classes  
test: C:\Users\Anuraag Shukla\Downloads\Vision Model Selection\Rice_Disease_Dataset\Rice Disease Dataset_split\test... found 465 images in 19 classes  
test: Fast image access  (ping: 0.00.0 ms, read: 1476.21076.1 MB/s, size: 360.7 KB)
test: Scanning C:\Users\Anuraag Shukla\Downloads\Vision Model Selection\Rice_Disease_Dataset\Rice Disease Dataset_split\test... 465 images, 0 corrupt: 100% ━━━━━━━━━━━━ 465/465  0.0s
test: C:\Users\Anuraag Shukla\Downloads\Vision Model Selection\Rice_Disease_Dataset\Ric

In [ ]:
# YOLOv11[For Cropping] + YOLO26-L[For Classification]
# Trained on Plant_Doc+Plant_Wild dataset[10 Classes] with basic transformations and equalization applied for 100 epochs with patience=30
# Tested on the test set provided to us to verify accuracy

In [ ]:
test_transform = [
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]

model_loaded = YOLO("runs/classify/train13/weights/best.pt")
metrics_testing = model_loaded.val(data="Merged_Dataset[Plant_Wild+Plant_Doc][10 Classes][For YOLO26]/train_split",split='test',device=0,augmentations=test_transform)
print(metrics_testing.top1)
print(metrics_testing.top5)

Ultralytics 8.4.21  Python-3.11.14 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3070 Ti, 8192MiB)
YOLO26l-cls summary (fused): 94 layers, 12,832,523 parameters, 0 gradients, 49.3 GFLOPs
train: C:\Users\Anuraag Shukla\Downloads\Vision Model Selection\Merged_Dataset[Plant_Wild+Plant_Doc][10 Classes][For YOLO26]\train_split\train... found 2845 images in 11 classes  
val: C:\Users\Anuraag Shukla\Downloads\Vision Model Selection\Merged_Dataset[Plant_Wild+Plant_Doc][10 Classes][For YOLO26]\train_split\val... found 717 images in 11 classes  
ERROR test: C:\Users\Anuraag Shukla\Downloads\Vision Model Selection\Merged_Dataset[Plant_Wild+Plant_Doc][10 Classes][For YOLO26]\train_split\test... found 842 images in 10 classes (requires 11 classes, not 10)
test: Fast image access  (ping: 0.10.0 ms, read: 147.595.5 MB/s, size: 64.8 KB)
test: Scanning C:\Users\Anuraag Shukla\Downloads\Vision Model Selection\Merged_Dataset[Plant_Wild+Plant_Doc][10 Classes][For YOLO26]\train_split\test... 842 images, 0 

# **Best Models**

In [ ]:
# YOLO v11 + ConvexNet-Base (Pipeline) 
# Trained on Plant_Doc + Plant_Wild Dataset for 20 epochs, and included a new transformation during training transforms.equalize()
# Tested on the dataset provided consisting 10 classes (excluding Tomato_Mosaic Virus, Tomato_Spider_Mites and Tomato_Target_Spot)

In [61]:
model_loaded = torch.load("ConVexNet-Base-Merged-Plant_Doc_Wild.pth",weights_only=False)

In [63]:
# Testing Block

full_dataset = datasets.ImageFolder("Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_testing" 

y_true,y_pred = final_test(model_loaded,labels,preprocess, TEST_DIR)

Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:03<00:00, 25.32it/s]

{'Potato_Early_blight': {'correct': 47, 'total': 87}, 'Potato_healthy': {'correct': 50, 'total': 63}, 'Potato_Lateblight': {'correct': 47, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 65, 'total': 100}, 'Tomato_Early_blight': {'correct': 31, 'total': 62}, 'Tomato_healthy': {'correct': 49, 'total': 71}, 'Tomato_Late_blight': {'correct': 54, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 56, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 50, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 58, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    54.02%
Potato_healthy                 |    79.37%
Potato_Lateblight              |    45.19%
Tomato_Bacterial_spot          |    65.00%
Tomato_Early_blight            |    50.00%
Tomato_healthy                 |    69.01%
Tomato_Late_blight             |    60.00%
Tomato_Leaf_mold               |    76.71%
Tomato_Septoria_leaf_s

In [ ]:
# Classification Report [Tomato Mosaic Virus is not present in the test set]
print(classification_report(y_true, y_pred,zero_division=0))

                                      precision    recall  f1-score   support

                 Potato_Early_blight       0.51      0.54      0.52        87
                   Potato_Lateblight       0.59      0.45      0.51       104
                      Potato_healthy       0.78      0.79      0.79        63
               Tomato_Bacterial_spot       0.58      0.65      0.61       100
                 Tomato_Early_blight       0.42      0.50      0.46        62
                  Tomato_Late_blight       0.60      0.60      0.60        90
                    Tomato_Leaf_mold       0.84      0.77      0.80        73
           Tomato_Septoria_leaf_spot       0.63      0.46      0.53       108
Tomato_Tomato_Yellow_Leaf_Curl_Virus       0.72      0.69      0.71        84
                      Tomato_healthy       0.65      0.69      0.67        71
                 Tomato_mosaic_virus       0.00      0.00      0.00         0

                            accuracy                          

In [ ]:
# ConvexNet-Base
# Trained on Rice Disease Dataset shared with us with basic transformations applied for 20 epochs
# Tested on the dataset provided to us to verify accuracy

In [57]:
model_loaded = torch.load("ConvexNet-Base-Rice-Disease-20-Epoch.pth",weights_only=False)

In [59]:
# Testing Block

full_dataset = datasets.ImageFolder("Rice_Disease_Dataset/Rice Disease Dataset_split/train")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.Resize((224,224)),
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Rice_Disease_Dataset/Rice Disease Dataset_split/test" 

y_true, y_pred = final_test(model_loaded,labels,preprocess, TEST_DIR)

Running validation on subset...


Testing Tungro: 100%|██████████| 30/30 [00:01<00:00, 16.38it/s]

{'Bacterial Leaf Blight': {'correct': 30, 'total': 30}, 'Bacterial Streak': {'correct': 14, 'total': 15}, 'Bakanae': {'correct': 15, 'total': 15}, 'Brown Spot': {'correct': 23, 'total': 30}, 'False Smut': {'correct': 15, 'total': 15}, 'Grassy Stunt Virus': {'correct': 12, 'total': 15}, 'Healthy Leaf': {'correct': 30, 'total': 30}, 'Hispa': {'correct': 28, 'total': 30}, 'Insect Affected': {'correct': 29, 'total': 30}, 'Leaf Blast': {'correct': 30, 'total': 30}, 'Leaf Scald': {'correct': 29, 'total': 30}, 'Leaf Smut': {'correct': 27, 'total': 30}, 'Narrow Brown Spot': {'correct': 30, 'total': 30}, 'Neck Blast': {'correct': 30, 'total': 30}, 'Ragged Stunt Virus': {'correct': 14, 'total': 15}, 'Sheath Blight': {'correct': 30, 'total': 30}, 'Sheath Rot': {'correct': 9, 'total': 15}, 'Stem Rot': {'correct': 13, 'total': 15}, 'Tungro': {'correct': 30, 'total': 30}}

Class Name                     | Accuracy  
---------------------------------------------
Bacterial Leaf Blight          |   100

In [60]:
# Classification Report
print(classification_report(y_true,y_pred))

                       precision    recall  f1-score   support

Bacterial Leaf Blight       0.86      1.00      0.92        30
     Bacterial Streak       1.00      0.93      0.97        15
              Bakanae       0.94      1.00      0.97        15
           Brown Spot       0.79      0.77      0.78        30
           False Smut       0.88      1.00      0.94        15
   Grassy Stunt Virus       1.00      0.80      0.89        15
         Healthy Leaf       0.88      1.00      0.94        30
                Hispa       1.00      0.93      0.97        30
      Insect Affected       1.00      0.97      0.98        30
           Leaf Blast       1.00      1.00      1.00        30
           Leaf Scald       1.00      0.97      0.98        30
            Leaf Smut       1.00      0.90      0.95        30
    Narrow Brown Spot       0.81      1.00      0.90        30
           Neck Blast       1.00      1.00      1.00        30
   Ragged Stunt Virus       0.88      0.93      0.90  